<a href="https://colab.research.google.com/github/DipaliKhochare/Website_Health_Monitor/blob/main/website_health_monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
%%writefile monitor.py
import socket, ssl, time, requests
from datetime import datetime, timezone

def check(domain):
    result = {"domain": domain}
    try:
        result["ip"] = socket.gethostbyname(domain)               # DNS
        with socket.create_connection((domain, 443), timeout=5) as sock:  # TCP
            ctx = ssl.create_default_context()
            with ctx.wrap_socket(sock, server_hostname=domain) as s:      # TLS
                cert = s.getpeercert()
                exp = datetime.strptime(cert["notAfter"], "%b %d %H:%M:%S %Y %Z")
                result["cert_days_left"] = (exp - datetime.utcnow()).days
        start = time.time()
        r = requests.get(f"https://{domain}", timeout=5)          # HTTP
        result["status"] = r.status_code
        result["latency_ms"] = round((time.time() - start) * 1000)
    except Exception as e:
        result["error"] = str(e)
    return result

if __name__ == "__main__":
    with open("sites.txt") as f:
        for line in f:
            if line.strip():
                print(datetime.now(timezone.utc).isoformat(), check(line.strip()))

Overwriting monitor.py


In [24]:
%%writefile sites.txt
google.com
github.com
example.com

Overwriting sites.txt


In [25]:
%%writefile run.sh
#!/bin/bash
mkdir -p logs
python3 monitor.py >> logs/health.log 2>&1

Overwriting run.sh


In [20]:
!chmod +x run.sh

In [26]:
!./run.sh

In [27]:
!cat logs/health.log

python3: can't open file '/content/monitor.py': [Errno 2] No such file or directory
  File "/content/monitor.py", line 5
    result = {
             ^
SyntaxError: '{' was never closed
/content/monitor.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  result["cert_days_left"] = (exp - datetime.utcnow()).days
2026-10-21T18:54:58.767191+00:00 {'domain': 'google.com', 'ip': '192.178.219.113', 'cert_days_left': 60, 'status': 200, 'latency_ms': 82}
2026-10-21T18:54:58.902301+00:00 {'domain': 'github.com', 'ip': '140.82.113.3', 'cert_days_left': 39, 'status': 200, 'latency_ms': 92}
2026-10-21T18:54:59.067195+00:00 {'domain': 'example.com', 'ip': '104.20.23.154', 'cert_days_left': 65, 'status': 200, 'latency_ms': 53}
/content/monitor.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a fu